## Train Generative Retrieval Model (T5)

Trains the T5 seq2seq model to predict the next item's Semantic ID from a user's interaction history.

**What this notebook does:**
1. Load Semantic ID dataset
2. Initialize T5 (~14.5M params)
3. Train with LR warmup + inverse sqrt decay (100k steps)
4. Evaluate with beam search → Recall@K, NDCG@K

In [ ]:
import sys

if "../" not in sys.path:
    sys.path.insert(0, "../")

from pathlib import Path
from typing import Iterator

import torch
import numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm

from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator

from tiger.dataset import TigerDataset
from tiger.model import create_model
from tiger.utils import get_device, set_seed

In [ ]:
# Paths
DATA_DIR = Path("../data/2014/processed")
SPLITS_PATH = DATA_DIR / "splits.parquet"
SEMANTIC_IDS_PATH = Path("../checkpoints/rqvae/semantic_ids.pt")
OUTPUT_DIR = Path("../checkpoints/model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset
SLIDING_WINDOW = True
MAX_SEQ_LEN = 20

# Model architecture
D_MODEL = 384
D_KV = 64
D_FF = 1024
NUM_LAYERS = 4
NUM_HEADS = 6
DROPOUT_RATE = 0.1

# Training
BATCH_SIZE = 256
NUM_STEPS = 100_000
LEARNING_RATE = 0.01
WARMUP_STEPS = 10_000
EVAL_EVERY = 5_000
SAVE_EVERY = 10_000

# Inference
BEAM_SIZE = 20

SEED = 42

In [ ]:
set_seed(SEED)
device = get_device()

### Load Dataset

Two dataloaders: train (with sliding window) and validation (single target).

Train uses sliding window to create ~95k examples from 19k users by using all sub-prefixes
of each user's history. Val always uses the full training history to predict the held-out val item.

In [ ]:
train_dataset = TigerDataset(
    splits_path=SPLITS_PATH,
    semantic_ids_path=SEMANTIC_IDS_PATH,
    split="train",
    max_seq_len=MAX_SEQ_LEN,
    sliding_window=SLIDING_WINDOW,
)

val_dataset = TigerDataset(
    splits_path=SPLITS_PATH,
    semantic_ids_path=SEMANTIC_IDS_PATH,
    split="val",
    max_seq_len=MAX_SEQ_LEN,
)


print(f"Train samples: {len(train_dataset):_}")
print(f"Val samples: {len(val_dataset):_}")
print(f"Vocab size: {train_dataset.vocab_size:_}")
print(f"Pad token: {train_dataset.pad_token}")

### Create Model

T5ForConditionalGeneration (random weights). Architecture matches paper:
- 4 encoder layers, 4 decoder layers
- 6 attention heads, 64 dim per head (d_model=384)
- d_ff=1024, dropout=0.1
- ~14.5M parameters

In [ ]:
model = create_model(
    vocab_size=train_dataset.vocab_size,
    pad_token_id=train_dataset.pad_token,
    d_model=D_MODEL,
    d_kv=D_KV,
    d_ff=D_FF,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    dropout_rate=DROPOUT_RATE,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:_}")

### Optimizer and LR scheduler

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)


def inv_sqrt_schedule(step: int) -> float:
    if step < WARMUP_STEPS:
        return 1.0
    return (WARMUP_STEPS / step) ** 0.5


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=inv_sqrt_schedule)

### Training loop

- Train for 100k steps. 
- The training is step-based (not epoch-based). 
- We cycle through the dataloader indefinitely. 
- Every EVAL_EVERY steps, compute validation loss to monitor overfitting.

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=NUM_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    eval_strategy="steps",
    eval_steps=EVAL_EVERY,
    save_steps=SAVE_EVERY,
    logging_steps=100,
    save_total_limit=3,
    predict_with_generate=False,
    report_to="none",
    remove_unused_columns=False,
)

In [ ]:
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)


def compute_metrics(eval_pred):
    preds, labels = eval_pred

    mask = labels != -100
    token_acc = (preds[mask] == labels[mask]).mean()

    seq_acc = ((preds == labels) | ~mask).all(axis=1).mean()

    return {
        "token_accuracy": float(token_acc),
        "sequence_accuracy": float(seq_acc),
    }

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    optimizers=(optimizer, scheduler),
)

trainer.train()